# STA 554 HW 9
Jillian Greene

This dataset (CH4_ML_dataset_aqua.csv) is the data I used in my master's thesis to develop a model predicting lake methane emissions based on remotely sensed environmental parameters. See Greene et al. (2026) *Limnology & Oceanography Letters* if you're interested to know more!

# Data set up:

In [1]:
import pandas as pd
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/13 10:13:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
ch4_data = pd.read_csv("CH4_ML_dataset_1d_aqua.csv")
ch4_data.head()

,Date,Latitude,Longitude,Site,Temp,rh,CH4_flux,Chla.x,pH,CI,...,TSM,pr,pr3,pr5,pr7,tmmx,vs,Lake,water_temp_K,sim_water
0,2024-05-04,43.223700,-86.28820,Muskegon - Channel,14.7,NaN,0.397998,0.70,8.5,0.0,...,0.527509,7.500459,16.740398,NaN,NaN,298.007956,4.100000,Muskegon,290.16881,290.16881
1,2024-05-04,43.223700,-86.28820,Muskegon - Channel,14.7,NaN,-0.586272,0.70,8.5,0.0,...,0.527509,7.500459,16.740398,NaN,NaN,298.007956,4.100000,Muskegon,290.16881,290.16881
2,2024-05-04,43.223700,-86.28820,Muskegon - Channel,14.7,NaN,0.965204,0.70,8.5,0.0,...,0.527509,7.500459,16.740398,NaN,NaN,298.007956,4.100000,Muskegon,290.16881,290.16881
3,2024-05-04,43.235767,-86.26263,Muskegon - Deep,15.1,NaN,2.625132,0.97,8.4,0.0,...,0.401975,7.120061,15.321439,NaN,NaN,298.042804,4.114472,Muskegon,290.16881,290.16881
4,2024-05-04,43.235767,-86.26263,Muskegon - Deep,15.1,NaN,2.133823,0.97,8.4,0.0,...,0.401975,7.120061,15.321439,NaN,NaN,298.042804,4.114472,Muskegon,290.16881,290.16881


In [3]:
ch4_data = spark.createDataFrame(ch4_data)
ch4_data.show(5)

26/04/13 10:13:37 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----------+---------+---------+------------------+----+---+------------+------+---+---+-----------+-----------+------------+--------------+--------------+--------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+-----------+-----------+----------------+----------------+---+---+----------------+----------------+--------+------------+---------+
|      Date| Latitude|Longitude|              Site|Temp| rh|    CH4_flux|Chla.x| pH| CI|     Chla.y|        PAR|       Kd490|         rtoa1|         rtoa2|         rtoa3|          rtoa4|          rtoa5|          rtoa6|          rtoa7|          rtoa8|          rtoa9|         rtoa10|         rtoa11|         rtoa12|     ADG443|        TSM|              pr|             pr3|pr5|pr7|            tmmx|              vs|    Lake|water_temp_K|sim_water|
+----------+---------+---------+------------------+----+---+------------+------+---+---+-----------+------

# Transformations:

In [5]:
from pyspark.ml.feature import SQLTransformer, VectorAssembler

# Set up transformation objects

# 1: Standardize CH4_Flux (log transform) but avoiding NaNs with negatives
sql_logtrans = SQLTransformer(
    statement="""
        SELECT *,
        LOG(1 + CASE 
                    WHEN CH4_Flux < 0 THEN 0
                    ELSE CH4_Flux
                END) AS CH4_Flux_log
        FROM __THIS__
    """
)

# 2: Select desired cols and set CH4_flux_log to 'label'
sql_select = SQLTransformer(
    statement = """
                SELECT Site, CI, `Chla.y` as Chla, PAR, Kd490, ADG443, TSM, pr, sim_water as Temp, CH4_Flux_log as label FROM __THIS__
                """
)

# 3: Convert 'Site' to binary based on site type (Vial vs Chamber)
sql_site_binary = SQLTransformer(
    statement = """
        SELECT *,
        CASE 
            WHEN Site RLIKE '^[0-9]+$' THEN 1.0
            ELSE 0.0
        END AS Site_binary
        FROM __THIS__
    """
)

# 4: Convert Temp from K to C
sql_temp_convert = SQLTransformer(
    statement = """
        SELECT *,
        (Temp - 273.15) AS Temp_C
        FROM __THIS__
    """
)

# Assembler
assembler = VectorAssembler(inputCols = ["Site_binary", "CI", "Chla", "PAR", "Kd490", "ADG443", "TSM", "pr", "Temp"],
                            outputCol = "features", handleInvalid = 'keep')

I have selected a basic linear regression model, random forest regression, and gradient boosted tree regression. The linear regression simply looks for linear relationships between the predictor variables (`features`) and the output (`label`). The RF and GBR are both decision tree models that split the data into different bins (trees) and use those to decide if the `features` lead down one tree, `label` must be X. Models usually have > 200 trees where the outputs are all averaged into one final decision. Furthermore, in the GBR, error-learning is incorporated.  

*In the following code, I had to reduce the grid size and number of folds to avoid memory issues!*

# Pipelines:

In [6]:
# Set up the train/test split 
train, test = ch4_data.randomSplit([0.8,0.2], seed = 120301)

In [7]:
# Pipelines
from pyspark.ml import Pipeline
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor
from pyspark.ml.tuning import ParamGridBuilder

# 1. Linear Regression
lr = LinearRegression()

paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0.01, 0.04, 0.06, 0.1]) \
    .addGrid(lr.elasticNetParam, [0.5, 0.8, 0.9, 1]) \
    .build()

lr_pipeline = Pipeline(stages = [sql_logtrans, sql_select, sql_site_binary, sql_temp_convert, assembler, lr])

# 2. Random Forest
rf = RandomForestRegressor()

rf_paramGrid = ParamGridBuilder() \
    .addGrid(rf.numTrees, [100, 200]) \
    .addGrid(rf.maxDepth, [5, 7]) \
    .build()

rf_pipeline = Pipeline(stages = [sql_logtrans, sql_select, sql_site_binary, sql_temp_convert, assembler, rf])

# 3. Gradient Boosted
gbr = GBTRegressor()

gbr_paramGrid = ParamGridBuilder() \
    .addGrid(gbr.maxIter, [50, 100]) \
    .addGrid(gbr.maxDepth, [3, 5]) \
    .build()

gbr_pipeline = Pipeline(stages = [sql_logtrans, sql_select, sql_site_binary, sql_temp_convert, assembler, gbr])


In [8]:
# Set up crossval object
from pyspark.ml.tuning import CrossValidator
from pyspark.ml.evaluation import RegressionEvaluator

lr_crossval = CrossValidator(estimator = lr_pipeline,
                          estimatorParamMaps = paramGrid,
                          evaluator = RegressionEvaluator(metricName = "mae"),
                          numFolds=5)

rf_crossval = CrossValidator(estimator = rf_pipeline,
                          estimatorParamMaps = rf_paramGrid,
                          evaluator = RegressionEvaluator(metricName = "mae"),
                          numFolds=2)

gbr_crossval = CrossValidator(estimator = gbr_pipeline,
                          estimatorParamMaps = gbr_paramGrid,
                          evaluator = RegressionEvaluator(metricName = "mae"),
                          numFolds=2)

In [9]:
# Run cross-validation on train

# Cache train
train = train.cache()
train.count()  # forces materialization

# LR
lr_cvModel = lr_crossval.fit(train)

# RF
rf_cvModel = rf_crossval.fit(train)

# GBR
gbr_cvModel = gbr_crossval.fit(train)

26/04/13 10:16:26 WARN DAGScheduler: Broadcasting large task binary with size 1239.9 KiB
26/04/13 10:16:27 WARN DAGScheduler: Broadcasting large task binary with size 1576.2 KiB
26/04/13 10:16:44 WARN DAGScheduler: Broadcasting large task binary with size 1152.7 KiB
26/04/13 10:16:45 WARN DAGScheduler: Broadcasting large task binary with size 1484.8 KiB
26/04/13 10:16:50 WARN DAGScheduler: Broadcasting large task binary with size 1065.3 KiB


# Testing:

In [13]:
# Crossval on test

# LR
lr_cvModel.transform(test).select("features", "label", "prediction").show(5)
lr_test_error = RegressionEvaluator(metricName = "mae").evaluate(lr_cvModel.transform(test))
print("LR:", lr_test_error)

# RF
rf_cvModel.transform(test).select("features", "label", "prediction").show(5)
rf_test_error = RegressionEvaluator(metricName = "mae").evaluate(rf_cvModel.transform(test))
print("RF:", rf_test_error)

# GBR
gbr_cvModel.transform(test).select("features", "label", "prediction").show(5)
gbr_test_error = RegressionEvaluator(metricName = "mae").evaluate(gbr_cvModel.transform(test))
print("GBR:", gbr_test_error)

+--------------------+------------------+------------------+
|            features|             label|        prediction|
+--------------------+------------------+------------------+
|[0.0,0.0,0.988188...|0.6755960638504697|1.4805151968192796|
|[0.0,0.0,0.301181...|4.4051692302668695|2.3385233809663752|
|[0.0,0.0,0.301181...| 4.398937362151815|2.3385233809663752|
|[0.0,0.0,0.353455...| 4.399274405022227|2.2032700874479083|
|[0.0,0.0,0.974803...| 4.044275503590655| 1.785484371535242|
+--------------------+------------------+------------------+
only showing top 5 rows
LR: 1.036431697312859
+--------------------+------------------+------------------+
|            features|             label|        prediction|
+--------------------+------------------+------------------+
|[0.0,0.0,0.988188...|0.6755960638504697|0.4929098584303289|
|[0.0,0.0,0.301181...|4.4051692302668695| 4.083701572675201|
|[0.0,0.0,0.301181...| 4.398937362151815| 4.083701572675201|
|[0.0,0.0,0.353455...| 4.39927440502222

Based on the MAE of the log transformed CH4 flux predictions, the Gradient-Boosted Regression tree is the best performing model. Based on my previous research and model development, this makes sense and with a bit more parameter tuning, all the models would perform nearly identical. 